In [ ]:
import pathlib
import random

import sklearn.neighbors
import igraph as ig
import leidenalg

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from bokeh.palettes import Category20
from bokeh.plotting import show
from bokeh.io import output_notebook
from bokeh.resources import INLINE

from IPython.display import display, Markdown

import trapallada.plot
import trapallada.analysis

In [ ]:
output_notebook(resources=INLINE)
# output_notebook()

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [ ]:
input_files_stem = '../results/epochs/20000/4_sources/epochs'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs_05_x_sd'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs' # rotated "C" below

It is turned into a `Pathlib`

In [ ]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

Files to *cache* the construction of the graph

In [ ]:
nodes_file = pathlib.Path('nodes.parquet')
edges_file = pathlib.Path('edges.parquet')

# Processing

For the sake of convenience

In [ ]:
zs_columns = [f'z_{i}' for i in range(1, parameters['model']['dim_z']+1)]
zs_columns[:7]

The *adjacency* matrix

In [ ]:
adj_matrix = sklearn.neighbors.kneighbors_graph(analysis_df[zs_columns].values, n_neighbors=10, mode='distance')
adj_matrix.shape

**Similarity** metric (as suggested by JCS)

In [ ]:
alpha = 1.
adj_matrix.data = np.exp(- alpha * adj_matrix.data)

A function to build an [igraph](https://igraph.org/) graph from a given similarity matrix

In [ ]:
def similarity_matix_to_graph(M: csr_matrix) -> ig.Graph:
    matrix_coo = M.tocoo()
    edges = list(zip(matrix_coo.row, matrix_coo.col))
    weights = matrix_coo.data

    g = ig.Graph(edges=edges, directed=False)
    g.es['weight'] = weights

    return g

G = similarity_matix_to_graph(adj_matrix)

Leidgen algorithm using [leidenalg](https://leidenalg.readthedocs.io/en/stable/index.html) implementation.

In [ ]:
partition = leidenalg.find_partition(G, leidenalg.ModularityVertexPartition)
partition

Commnities are *extracted* from the resulting partition, and its number computed

In [ ]:
communities = partition.membership
n_communities = max(communities) + 1
n_communities

In [ ]:
# nodes_df = pd.read_parquet(nodes_file)
# edges_df = pd.read_parquet(edges_file)

The *layout* of the nodes when plotting the graph

In [ ]:
# so that, for the same data, the results are consistent
random.seed(42)

layout = G.layout('auto')
layout

A `DataFrame` with information on the nodes

In [ ]:
palette = Category20[20]
nodes_df = pd.DataFrame(
    {
        'community': communities,
        # 'fill_color': [palette[c ]for c in communities],
        'fill_color': [palette[c % 20]for c in communities],
        'coordinates': layout.coords
    },
    index=analysis_df.index
)
unknown_series = analysis_df['main_type'].isna()
nodes_df.loc[unknown_series, 'marker'] = 'square'
nodes_df.loc[~unknown_series, 'marker'] = 'diamond'
nodes_df

Another one for the edges

In [ ]:
edges_df = pd.DataFrame(
    [(edge.source, edge.target) for edge in G.es], columns=['start', 'end']
)
edges_df

The above `DataFrame`s are saved to disk

In [ ]:
nodes_df.to_parquet(nodes_file)
edges_df.to_parquet(edges_file)

# Overall view

In [ ]:
fig = trapallada.analysis.plot_graph(analysis_df, nodes_df, edges_df)
show(fig)

# Region 1

In [ ]:
fig.x_range.start, fig.x_range.end = 222, 228
fig.y_range.start, fig.y_range.end = -170, -162
show(fig)

In [ ]:
for i in [22444, 14235, 48650]:
    # display(Markdown(f'### {i}'))
    print(analysis_df.filter(regex='^(?!z_|tsne_|set$)').loc[i])
    display(trapallada.analysis.predict_and_plot(model, ds, i, display_metadata=False))

# Region 2

In [ ]:
fig.x_range.start, fig.x_range.end = -159, -152
fig.y_range.start, fig.y_range.end = 44, 58

show(fig)

In [ ]:
for i in [23061, 5139, 7468]:
    display(Markdown(f'### {i}'))
    print(analysis_df.filter(regex='^(?!z_|tsne_|set$)').loc[i])
    display(trapallada.analysis.predict_and_plot(model, ds, i, display_metadata=False))